[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/19_Context_Module.ipynb)

# 上下文模块 —— 实用指南

Semantica 的 `context` 模块是让智能体具备"状态"的那一层。它结合了:

- 通过 `AgentMemory` 提供的**记忆**(短期 + 长期)
- 通过 `ContextGraph` 提供的**图上下文**
- 通过 `ContextRetriever` 提供的**混合检索**(向量 + 记忆 + 图)
- 通过 `AgentContext` 提供的**高层用户体验**(推荐的入口)
- 通过 `EntityLinker` 提供的**实体链接**
- 通过 `registry` 和 `config` 提供的**扩展性与配置**

本 Notebook 聚焦于小而可运行的示例,并将导入限定在各自的单元格内。

In [ ]:
!pip install -q semantica

## 1)向量库(用于长期记忆)

`VectorStore` 可以通过其内部的嵌入器生成嵌入。如果你的环境中没有可用的嵌入器,它会回退到随机向量,使 API 在演示中仍然可用。

In [ ]:
from semantica.vector_store import VectorStore

vs = VectorStore(backend="inmemory", dimension=384)

if getattr(vs, "embedder", None) and hasattr(vs.embedder, "set_text_model"):
    vs.embedder.set_text_model(method="fastembed", model_name="BAAI/bge-small-en-v1.5")

vs.backend, vs.dimension

## 2)使用 `AgentContext` 快速上手(推荐)

`AgentContext` 是用户友好的接口,将记忆、向量库和图联系在一起。如果你传入一个 `ContextGraph`,系统就可以进行 GraphRAG 风格的检索。

In [ ]:
from semantica.context import AgentContext, ContextGraph

kg = ContextGraph()
context = AgentContext(vector_store=vs, knowledge_graph=kg)

context.config

## 3)存储与检索记忆

单个字符串会被视为一个记忆条目。你可以通过对元数据友好的参数附加 `conversation_id` 和 `user_id`。

In [ ]:
memory_id = context.store(
    "User prefers short answers about Python.",
    conversation_id="conv_1",
    user_id="user_1",
    metadata={"type": "preference"},
)

context.get_memory(memory_id)

In [ ]:
context.store(
    "User is working on Semantica context module examples.",
    conversation_id="conv_1",
    user_id="user_1",
    metadata={"type": "note"},
)

context.retrieve("Python answers", max_results=3)

In [ ]:
context.conversation("conv_1", max_items=10)

## 4)导出、保存、加载

`AgentContext` 包含简单的持久化辅助方法。本示例使用一个临时目录。

In [ ]:
export_json = context.export(conversation_id="conv_1", format="json")
export_json[:300]

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as d:
    context.save(d)
    context.load(d)

context.conversation_summary("conv_1")

## 5)存储文档并构建上下文图

如果你存储的是一个列表,`AgentContext.store(...)` 会把它视为文档。为了让本 Notebook 保持轻量且结果确定,我们为每个文档传入预先抽取好的实体和关系。

In [ ]:
documents = [
    {
        "id": "doc_1",
        "content": "Python is used for machine learning.",
        "metadata": {"source": "docs"},
        "entities": [
            {"id": "e_python", "text": "Python", "type": "PROGRAMMING_LANGUAGE"},
            {"id": "e_ml", "text": "Machine Learning", "type": "CONCEPT"},
        ],
        "relationships": [
            {
                "source_id": "e_python",
                "target_id": "e_ml",
                "type": "used_for",
                "confidence": 0.9,
            }
        ],
    },
    {
        "id": "doc_2",
        "content": "PyTorch is a machine learning framework.",
        "metadata": {"source": "docs"},
        "entities": [
            {"id": "e_pytorch", "text": "PyTorch", "type": "FRAMEWORK"},
            {"id": "e_ml", "text": "Machine Learning", "type": "CONCEPT"},
        ],
        "relationships": [
            {
                "source_id": "e_pytorch",
                "target_id": "e_ml",
                "type": "implements",
                "confidence": 0.95,
            }
        ],
    },
]

stats = context.store(
    documents,
    extract_entities=False,
    extract_relationships=False,
    link_entities=True,
)

stats

In [ ]:
kg.stats()

## 6)使用 `ContextGraph` 探索图

该图支持关键词查询和邻居扩展。

In [ ]:
kg.query("machine learning")

In [ ]:
kg.get_neighbors("e_python", hops=2)

## 7)使用 `EntityLinker` 进行实体链接

`EntityLinker` 会分配稳定的 URI,并能链接跨来源的相关或重复实体。

In [ ]:
from semantica.context import EntityLinker

linker = EntityLinker(knowledge_graph={"entities": [{"id": "e_py", "text": "Python", "type": "PROGRAMMING_LANGUAGE"}]})

entities = [
    {"id": "e1", "text": "Python", "type": "PROGRAMMING_LANGUAGE"},
    {"id": "e2", "text": "PyTorch", "type": "FRAMEWORK"},
]

linked = linker.link("Python and PyTorch", entities=entities)
[(e.entity_id, e.uri, len(e.linked_entities)) for e in linked]

In [ ]:
linker.link_entities("e1", "e2", link_type="related_to", confidence=0.8)
linker.get_entity_links("e1")[:2]

In [ ]:
linker.build_entity_web()["statistics"]

## 8)底层构建块:`AgentMemory` 与 `ContextRetriever`

如果你想要比 `AgentContext` 更多的控制,可以直接组装这些部件。

In [ ]:
from semantica.context import AgentMemory, ContextRetriever

memory = AgentMemory(vector_store=vs, knowledge_graph=kg, retention_policy="unlimited")
memory.store("Python powers Semantica.", metadata={"type": "fact", "conversation_id": "conv_2"})

retriever = ContextRetriever(memory_store=memory, knowledge_graph=kg, vector_store=vs)
results = retriever.retrieve("Python Semantica", max_results=5)

[(r.content, r.source, round(r.score, 3)) for r in results]

## 9)方法、注册表与配置

`methods` 层暴露了便捷函数,而 `registry` 让你可以接入自己的实现。`config` 提供运行时配置。

In [ ]:
from semantica.context.config import context_config

context_config.set("retention_policy", "7_days")
context_config.get("retention_policy")

In [ ]:
from semantica.context.methods import build_context_graph
from semantica.context.registry import method_registry

def custom_graph_method(entities, relationships, conversations=None, **kwargs):
    return {
        "nodes": [],
        "edges": [],
        "statistics": {"node_count": 0, "edge_count": 0},
    }

method_registry.register("graph", "custom_demo", custom_graph_method)
method_registry.list_all("graph")

In [ ]:
build_context_graph(
    entities=[{"id": "e1", "text": "Python", "type": "PROGRAMMING_LANGUAGE"}],
    relationships=[{"source_id": "e1", "target_id": "e2", "type": "related_to"}],
    method="custom_demo",
)